# Anomaly Detection & Time Series - Assignment DA-AG-018
Run in Google Colab. Q7 needs `train.csv` from Kaggle *New York City Taxi Fare Prediction*.

## Q6 - Decomposition of AirPassengers

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose

url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv"
df = pd.read_csv(url, parse_dates=["Month"], index_col="Month")
df.columns = ["Passengers"]
print(df.head())
print("Shape:", df.shape)

# 1. Plot the original series
df["Passengers"].plot(figsize=(10, 4), title="Monthly Airline Passengers (1949-1960)")
plt.ylabel("Passengers")
plt.show()

# 2. Decompose (multiplicative, because seasonal swings grow with the level)
result = seasonal_decompose(df["Passengers"], model="multiplicative", period=12)
fig = result.plot()
fig.set_size_inches(10, 8)
plt.tight_layout()
plt.show()

## Q7 - Isolation Forest on NYC Taxi Fare

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# NYC Taxi Fare data (Kaggle: new-york-city-taxi-fare-prediction, train.csv)
df = pd.read_csv("train.csv", nrows=100000).dropna()

# Keep valid NYC coordinates and sensible fares
df = df[(df.pickup_latitude.between(40, 42)) & (df.dropoff_latitude.between(40, 42)) &
        (df.pickup_longitude.between(-75, -72)) & (df.dropoff_longitude.between(-75, -72)) &
        (df.fare_amount > 0)]

# Trip distance in km (haversine formula)
def haversine(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

df["distance_km"] = haversine(df.pickup_latitude, df.pickup_longitude,
                              df.dropoff_latitude, df.dropoff_longitude)

X = df[["distance_km", "fare_amount"]]
X_scaled = StandardScaler().fit_transform(X)

iso = IsolationForest(n_estimators=200, contamination=0.01, random_state=42)
df["anomaly"] = iso.fit_predict(X_scaled)          # -1 = anomaly, 1 = normal
print(df["anomaly"].value_counts())

plt.figure(figsize=(9, 6))
normal, anom = df[df.anomaly == 1], df[df.anomaly == -1]
plt.scatter(normal.distance_km, normal.fare_amount, s=4, c="steelblue", label="Normal")
plt.scatter(anom.distance_km, anom.fare_amount, s=14, c="red", label="Anomaly")
plt.xlabel("Trip distance (km)"); plt.ylabel("Fare amount ($)")
plt.title("Isolation Forest - NYC Taxi Fare Anomalies")
plt.legend(); plt.show()

## Q8 - SARIMA forecast

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX

url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv"
df = pd.read_csv(url, parse_dates=["Month"], index_col="Month")
y = df["Passengers"].asfreq("MS")

# SARIMA(1,1,1)(1,1,1,12): d=1 and D=1 remove trend and yearly seasonality
model = SARIMAX(y, order=(1, 1, 1), seasonal_order=(1, 1, 1, 12))
fit = model.fit(disp=False)
print(fit.summary())

# Forecast next 12 months with 95% confidence interval
fc = fit.get_forecast(steps=12)
mean, ci = fc.predicted_mean, fc.conf_int()

plt.figure(figsize=(10, 5))
plt.plot(y, label="Observed")
plt.plot(mean, color="red", label="Forecast (next 12 months)")
plt.fill_between(ci.index, ci.iloc[:, 0], ci.iloc[:, 1], color="pink", alpha=0.4, label="95% CI")
plt.title("SARIMA Forecast - Airline Passengers")
plt.xlabel("Date"); plt.ylabel("Passengers"); plt.legend(); plt.show()
print(mean.round(0))

## Q9 - Local Outlier Factor

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import LocalOutlierFactor

# Numerical dataset: Wine (178 samples, 13 numeric features)
X = load_wine().data
X_scaled = StandardScaler().fit_transform(X)

lof = LocalOutlierFactor(n_neighbors=20, contamination=0.05)
labels = lof.fit_predict(X_scaled)                 # -1 = outlier, 1 = inlier
scores = -lof.negative_outlier_factor_             # higher = more anomalous
print("Total points:", len(X), "| Outliers found:", (labels == -1).sum())
print("Top 5 LOF scores:", np.sort(scores)[-5:].round(2))

# 2D view using PCA
X2 = PCA(n_components=2).fit_transform(X_scaled)
plt.figure(figsize=(8, 6))
plt.scatter(X2[labels == 1, 0], X2[labels == 1, 1], c="steelblue", s=30, label="Inlier")
plt.scatter(X2[labels == -1, 0], X2[labels == -1, 1], c="red", s=80,
            edgecolors="k", label="Outlier")
plt.xlabel("PC 1"); plt.ylabel("PC 2")
plt.title("Local Outlier Factor - Wine dataset")
plt.legend(); plt.show()

## Q10 - Power grid: forecasting + anomaly detection

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from statsmodels.tsa.statespace.sarimax import SARIMAX

# ---- 1. Simulated 15-minute grid data (replace with real feed) ----
rng = np.random.default_rng(42)
idx = pd.date_range("2025-01-01", periods=96 * 14, freq="15min")        # 14 days
hour = idx.hour + idx.minute / 60
temp = 25 + 8 * np.sin((hour - 9) / 24 * 2 * np.pi) + rng.normal(0, 0.5, len(idx))
usage = 300 + 80 * np.sin((hour - 6) / 24 * 2 * np.pi) + 4 * temp + rng.normal(0, 8, len(idx))
usage[[200, 600, 900]] += [250, -200, 300]                              # injected spikes/drops
data = pd.DataFrame({"usage": usage, "temp": temp}, index=idx)

# ---- 2. Short-term forecasting: SARIMAX with weather as exogenous input ----
train, test = data.iloc[:-96], data.iloc[-96:]                          # forecast last day
model = SARIMAX(train["usage"], exog=train[["temp"]],
                order=(2, 0, 1), seasonal_order=(1, 0, 0, 96))
fit = model.fit(disp=False, maxiter=50)
pred = fit.get_forecast(steps=96, exog=test[["temp"]]).predicted_mean
mape = (abs(test["usage"] - pred) / test["usage"]).mean() * 100
print(f"Next-24h forecast MAPE: {mape:.2f}%")

# ---- 3. Anomaly detection on model residuals + rolling window ----
feat = pd.DataFrame({"resid": fit.resid,
                     "roll_mean": data["usage"].rolling(4).mean().loc[train.index],
                     "diff": data["usage"].diff().loc[train.index]}).dropna()
iso = IsolationForest(contamination=0.005, random_state=42).fit(feat)
feat["anomaly"] = iso.predict(feat)
print("Anomalies flagged:", (feat.anomaly == -1).sum())

# ---- 4. Visualise ----
fig, ax = plt.subplots(2, 1, figsize=(11, 7))
ax[0].plot(test["usage"], label="Actual"); ax[0].plot(pred, "r--", label="Forecast")
ax[0].set_title("Next-day demand forecast (SARIMAX)"); ax[0].legend()
ax[1].plot(data["usage"], lw=0.7)
a = feat[feat.anomaly == -1].index
ax[1].scatter(a, data.loc[a, "usage"], c="red", label="Anomaly")
ax[1].set_title("Detected spikes/drops (Isolation Forest)"); ax[1].legend()
plt.tight_layout(); plt.show()